In [84]:
import pandas as pd
import re
from datetime import datetime
from dateutil import parser
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from datetime import datetime, timedelta
import seaborn as sns
import numpy as np
from scipy.optimize import minimize_scalar


In [86]:
# Reference time for all power measurements
reference_time = datetime(1970, 1, 1, 0, 0, 0)



In [88]:


def parse_log_with_event_type_precise_ms(log_content):
    """
    Parse the Selenium log file and prioritize explicit durations in milliseconds when available.
    """
    events = []
    current_event = {}
    current_action = None

    for line in log_content:
        # Detect Fibonacci execution
        if "fibonacci" in line.lower():
            current_event['event_type'] = "Fibonacci"
            if "start" in line.lower():
                match = re.search(r"start.*\[(.*)\]", line)
                if match:
                    current_event['start_time'] = parser.parse(match.group(1))
            elif "end" in line.lower():
                match = re.search(r"end.*\[(.*)\]", line)
                if match:
                    current_event['end_time'] = parser.parse(match.group(1))
                    # Calculate duration in ms if no explicit duration is provided
                    if 'start_time' in current_event and 'end_time' in current_event and 'duration' not in current_event:
                        duration = (current_event['end_time'] - current_event['start_time']).total_seconds() * 1000
                        current_event['duration'] = int(duration)
                        # print(f"Parsed Fibonacci duration: {current_event['duration']} ms (calculated)")  # Debugging
                        events.append(current_event)
                        current_event = {}

        # Detect action runs
        # Detect action file path
        elif re.search(r"^\./", line):
            # Match action file paths with possible wildcards
            match = re.search(r"\./([^;]+)", line, re.IGNORECASE)
            if match:
                current_action = match.group(1).strip()
                

        elif "start test" in line.lower():
            match = re.search(r"start.*\[(.*)\]", line)
            if match:
                current_event = {'event_type': current_action}
                current_event['start_time'] = parser.parse(match.group(1))

        elif "end test" in line.lower():
            match = re.search(r"end.*\[(.*)\]", line)
            if match:
                current_event['end_time'] = parser.parse(match.group(1))
                # Calculate duration in ms if no explicit duration is provided
                if 'start_time' in current_event and 'end_time' in current_event and 'duration' not in current_event:
                    duration = (current_event['end_time'] - current_event['start_time']).total_seconds() * 1000
                    current_event['duration'] = int(duration)
                    # print(f"Parsed Action duration: {current_event['duration']} ms (calculated)")  # Debugging

        elif "duration of test" in line.lower():
            match = re.search(r"duration of test (\d+)", line)
            if match:
                # Use explicit duration in milliseconds
                current_event['duration'] = int(match.group(1))
                # print(f"Parsed explicit duration: {current_event['duration']} ms (explicit)")  # Debugging

        # Append the current event if fully parsed
        if 'start_time' in current_event and 'end_time' in current_event and 'duration' in current_event:
            events.append(current_event)
            current_event = {}

    return pd.DataFrame(events)




In [90]:
def extract_durations(dict_):
     # Load log file (replace with actual file path)
    log_file_path = dict_['logpath']
    with open(log_file_path, 'r') as f:
        log_content = f.readlines()

    # Parse log file
    log_events = parse_log_with_event_type_precise_ms(log_content)
    #display(log_events)
    
    return log_events

In [92]:
# Example usage
def extract_time_line(dict_):
    # Example configurations
    

    # Load log file (replace with actual file path)
    log_file_path = dict_['logpath']
    with open(log_file_path, 'r') as f:
        log_content = f.readlines()

    # Parse log file
    log_events = parse_log_with_event_type_precise_ms(log_content)
    #display(log_events)
    
    
    # Construct timeline
    timeline = construct_event_timeline_with_types_ms(dict_, log_events)

    # Define reference start time as 1970-01-01 00:00:00	

    reference_time = datetime(1970, 1, 1, 0, 0, 0)
        
    # Add a new column for datetime
    timeline['dateTime'] = timeline['duration (ms)'].apply(lambda ms: reference_time + timedelta(milliseconds=ms))

    # display(timeline.head())
    return log_events, timeline

# Run the program
#if __name__ == "__main__":
#    main()
# selenium dict test
# dict_ = all_dicts[8]
    
# Construct timeline for this framework (replace with actual timeline generation code)
# timeline = extract_time_line(dict_)

# print(timeline.head())


# Temperatures

In [126]:
import re
from dateutil import parser

def parse_log_with_event_type_and_temp(log_content):
    """
    Parse the log file, prioritize explicit durations in milliseconds when available,
    and also extract temperature readings before and after events into separate columns.
    """
    events = []
    current_event = {}
    current_action = None
    temp_before = None
    temp_after = None


    for line in log_content:
        # Detect action runs
        # Detect action file path
        if re.search(r"^\./", line):
            # Match action file paths with possible wildcards
            match = re.search(r"\./([^;]+)", line, re.IGNORECASE)
            if match:
                current_action = match.group(1).strip()
                current_event = {'event_type': current_action}
                
        # Detect temp_before line (before test starts)
        elif "temp_before" in line.lower():
            match = re.search(r"temp_before = temp=([0-9.]+)'C", line)
            if match:
                current_event['temp_before'] = float(match.group(1))
                
        # Detect temp_after line (after test ends)
        elif "temp_after" in line.lower():
            match = re.search(r"temp_after = temp=([0-9.]+)'C", line)
            if match:
                current_event['temp_after'] = float(match.group(1))   
        #print(current_event)

        elif "start test" in line.lower():
            match = re.search(r"start.*\[(.*)\]", line)
            if match:
                current_event['start_time'] = parser.parse(match.group(1))
                
        # Append the current event if fully parsed
        if 'temp_after' in current_event and 'temp_before' in current_event and 'start_time' in current_event:
            events.append(current_event)
            current_event = {}


    return pd.DataFrame(events)


In [122]:
def extract_temp_from_logs(dict_):
     # Load log file (replace with actual file path)
    log_file_path = dict_['logpath']
    with open(log_file_path, 'r') as f:
        log_content = f.readlines()

    # Parse log file
    log_events = parse_log_with_event_type_and_temp(log_content)
    #display(log_events)
    
    return log_events

In [99]:
#for dict_ in all_dicts:
#    print(dict_)